In [ ]:
from model_arch import *

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup

In [ ]:
BATCH_SIZE = 8

class TokenDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
        self.n = (len(data) - 1) // seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        start = i * self.seq_len
        chunk = torch.from_numpy(self.data[start : start + self.seq_len + 1].astype(np.int64))
        return chunk[:-1], chunk[1:]

data = np.fromfile("tokens_shuffled.bin", dtype=np.uint16)
split = int(len(data) * 0.999)
train_ds = TokenDataset(data[:split], 1024)
val_ds = TokenDataset(data[split:], 1024)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
cfg = LMConfig(vocab_size=50260)

model = DecoderLM(cfg).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=500, num_training_steps=(len(train_ds)//BATCH_SIZE))
use_fp16 = torch.cuda.is_available()
scaler = torch.amp.GradScaler(enabled=use_fp16)

In [ ]:
run_loss = 0.0
step = 0

for x, y in train_loader:
    model.train()
    x, y = x.to(DEVICE), y.to(DEVICE)
    optimizer.zero_grad()
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_fp16):
        logits = model(x)
        batch_size, seq_len, vocab_size = logits.shape
        logits = logits.reshape(batch_size*seq_len, vocab_size)
        y = y.reshape(batch_size*seq_len)
        loss = F.cross_entropy(logits, y)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    run_loss += loss.item()

    step += 1
    if step % 1000 == 0:
        print("train loss:", run_loss/1000)
        run_loss = 0.0

In [ ]:
torch.save(model.state_dict(), "checkpoint.pt")